<a href="https://colab.research.google.com/github/pallapunandhini456-dot/INFO-5731/blob/main/In_Class_Exercise_5%266_Feature_Extraction_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# In-Class Assignment — Python for Feature Extraction
**Time:** 20 minutes  |  **Points:** 20


....


Dataset file: `product_reviews.txt`

## Load the dataset
Download it from **Canvas**, then run the upload cell below to select the file from your computer.


In [4]:
from google.colab import files
uploaded = files.upload()

Saving product_reviews.txt to product_reviews.txt


## Q1 (2 point) — Load & Read & inspect

## Note about the header
**Important:** The dataset file includes a **header row** (column names).  
Make sure the header is used as the column names — it **should NOT appear as a data row** in your DataFrame.

Tip: Use the filename variable printed above when reading the file.
After Reading, check that your columns are `id` and `text`.
Print: **(a)** `df.shape`, **(b)** `df.head(3\)`.  


In [5]:
import pandas as pd

# load dataset
df = pd.read_csv('product_reviews.txt', sep='\t')

# print shape
print("Shape:", df.shape)

# show first 3 rows
df.head(3)



Shape: (10, 2)


,id,text
0,1,Love this blender! Smoothies are super creamy ...
1,2,Terrible quality... stopped working after 2 da...
2,3,Good value for the price. Shipping was quick.


## Q2 (4 points) — Basic handcrafted features  
Create these columns and then display the DataFrame:
- `word_count` = number of words  
- `char_count` = number of characters  
- `avg_word_len` = average word length (ignore punctuation)  
- `excl_count` = number of `!` characters  

Print: `id, word_count, char_count, avg_word_len, excl_count`.


In [6]:
import re

# word count
df['word_count'] = df['text'].apply(lambda x: len(x.split()))

# char count
df['char_count'] = df['text'].apply(len)

# avg word length (ignore punctuation)
def avg_word_len(text):
    words = re.findall(r'\b\w+\b', text)  # remove punctuation
    if len(words) == 0:
        return 0
    return sum(len(word) for word in words) / len(words)

df['avg_word_len'] = df['text'].apply(avg_word_len)

# exclamation count
df['excl_count'] = df['text'].apply(lambda x: x.count('!'))

# print required columns
df[['id', 'word_count', 'char_count', 'avg_word_len', 'excl_count']]



,id,word_count,char_count,avg_word_len,excl_count
0,1,9,55,5.000000,1
1,2,7,51,5.571429,3
2,3,8,45,4.500000,0
3,4,10,56,4.500000,0
4,5,8,53,5.500000,1
5,6,10,51,4.000000,0
6,7,9,50,4.444444,0
7,8,6,40,5.500000,0
8,9,7,52,6.285714,0
9,10,7,48,4.750000,0


## Q3 (6 points) — Bag-of-Words (CountVectorizer)  
Use `CountVectorizer(stop_words="english")` on `df["text"]`. Print:
1) vocabulary size (number of features)  
2) top 10 words by **total count** across all documents (word + count)


In [7]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

# create vectorizer
vectorizer = CountVectorizer(stop_words='english')

# fit and transform
X = vectorizer.fit_transform(df['text'])

# 1) vocabulary size
vocab_size = len(vectorizer.get_feature_names_out())
print("Vocabulary Size:", vocab_size)

# 2) top 10 words by total count
word_counts = X.sum(axis=0).A1   # sum counts across all rows
words = vectorizer.get_feature_names_out()

# combine words and counts
word_freq = list(zip(words, word_counts))

# sort descending
word_freq_sorted = sorted(word_freq, key=lambda x: x[1], reverse=True)

# print top 10
print("\nTop 10 words:")
for word, count in word_freq_sorted[:10]:
    print(word, ":", count)



Vocabulary Size: 50

Top 10 words:
amazing : 1
app : 1
battery : 1
blender : 1
box : 1
buy : 1
charged : 1
clear : 1
crashing : 1
creamy : 1


## Q4 (4 points) — Bigram features  
Use `CountVectorizer(stop_words="english", ngram_range=(2,2))`.  
Print the top 5 bigrams by total count (bigram + count).


In [8]:
from sklearn.feature_extraction.text import CountVectorizer

# bigram vectorizer
vectorizer2 = CountVectorizer(stop_words='english', ngram_range=(2,2))

# fit and transform
X2 = vectorizer2.fit_transform(df['text'])

# get bigram names
bigrams = vectorizer2.get_feature_names_out()

# total counts
bigram_counts = X2.sum(axis=0).A1

# combine
bigram_freq = list(zip(bigrams, bigram_counts))

# sort descending
bigram_sorted = sorted(bigram_freq, key=lambda x: x[1], reverse=True)

# print top 5
print("Top 5 bigrams:")
for bg, count in bigram_sorted[:5]:
    print(bg, ":", count)


Top 5 bigrams:
amazing screen : 1
app keeps : 1
battery life : 1
blender smoothies : 1
box damaged : 1


## Q5 (4 points) — TF-IDF features  
Use `TfidfVectorizer(stop_words="english", ngram_range=(1,2))`.  
Compute the **average TF-IDF** score of each term across documents and print the top 5 terms (term + avg score).


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# TF-IDF vectorizer (with unigrams + bigrams)
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1,2))

# fit and transform
X = tfidf.fit_transform(df['text'])

# get feature names
terms = tfidf.get_feature_names_out()

# calculate average TF-IDF score for each term
avg_scores = X.mean(axis=0).A1

# combine terms and scores
term_scores = list(zip(terms, avg_scores))

# sort descending
sorted_terms = sorted(term_scores, key=lambda x: x[1], reverse=True)

# print top 5
print("Top 5 TF-IDF terms:")
for term, score in sorted_terms[:5]:
    print(term, ":", round(score, 4))



Top 5 TF-IDF terms:
clear : 0.0378
easy : 0.0378
easy instructions : 0.0378
instructions : 0.0378
instructions clear : 0.0378


## Grading Checklist
- Q1: correct prints  
- Q2: correct feature columns + requested display  
- Q3: correct vocabulary size + correct top 10 words by total count  
- Q4: correct top 5 bigrams by total count  
- Q5: correct top 5 TF-IDF terms by average score
